[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/repaso_final/17_repaso_integrador.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Repaso integrador de las tres unidades

**Campos y Ondas Electromagnéticas (ICEE1033)**

Este notebook recorre las tres unidades del curso con un problema de cada
una. Sirve para repasar antes de una evaluación recuperativa, o simplemente
para comprobar que las tres partes del curso encajan.

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Resolver un problema de electrostática con simetría esférica (Unidad 1).
2. Resolver una interfaz en incidencia normal y verificar el balance de
   potencia (Unidad 2).
3. Resolver una línea de transmisión con carga desadaptada (Unidad 3).
4. Reconocer qué herramienta corresponde a cada tipo de problema.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "campos_electrostaticos.py", "interfaces_planas.py", "lineas_transmision.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from campos_electrostaticos import (
    carga_total_esfera_uniforme,
    campo_esfera_uniforme,
    potencial_esfera_uniforme,
)
from interfaces_planas import (
    coeficiente_reflexion_normal,
    coeficiente_transmision_normal,
    reflectancia_normal,
    transmitancia_normal,
)
from lineas_transmision import (
    coeficiente_reflexion,
    razon_onda_estacionaria,
    impedancia_entrada,
    longitud_electrica,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. El hilo que une las tres unidades

Puede parecer que el curso son tres temas sueltos. No lo son.

**Unidad 1** responde: dadas las cargas y corrientes, ¿cuáles son los campos?
Las herramientas son la ley de Gauss, la de Ampère y las condiciones de
borde.

**Unidad 2** responde: si a los campos se les deja evolucionar en el tiempo,
¿qué hacen? La respuesta es que se propagan como ondas. Y cuando esas ondas
llegan a una interfaz, las condiciones de borde de la Unidad 1 vuelven a
aparecer: de ahí salen Snell y Fresnel.

**Unidad 3** responde: ¿cómo se guía una onda para que llegue donde uno
quiere? Con líneas, guías y antenas. Y los parámetros de una línea coaxial,
$C'$ y $L'$, son exactamente los de la Unidad 1.

El mismo coeficiente de reflexión aparece en las tres:

$$
\Gamma = \frac{n_1 - n_2}{n_1 + n_2}
\qquad\text{y}\qquad
\Gamma = \frac{Z_L - Z_0}{Z_L + Z_0}
$$

son la misma fórmula. Una interfaz óptica y una carga desadaptada son el
mismo problema con otro nombre.

## 3. Ecuaciones de las tres rutas

**Ruta 1 — esfera con carga uniforme (Unidad 1):**

$$
Q = \rho_v\frac{4}{3}\pi a^{3},
\qquad
E(r) = \frac{Q}{4\pi\varepsilon_0 r^{2}},
\qquad
V(r) = \frac{Q}{4\pi\varepsilon_0 r}
\qquad (r \ge a).
$$

**Ruta 2 — interfaz en incidencia normal (Unidad 2):**

$$
\Gamma = \frac{n_1 - n_2}{n_1 + n_2},
\qquad
\tau = 1 + \Gamma,
\qquad
R + T = 1 .
$$

**Ruta 3 — línea de transmisión (Unidad 3):**

$$
\Gamma = \frac{Z_L - Z_0}{Z_L + Z_0},
\qquad
\mathrm{ROE} = \frac{1+|\Gamma|}{1-|\Gamma|},
\qquad
Z_{\text{in}} = Z_0\frac{Z_L + jZ_0\tan\beta l}{Z_0 + jZ_L\tan\beta l}.
$$

## 4. Qué revisar en cada ruta

**Ruta 1.** Fuera de la esfera todo se comporta como una carga puntual. La
relación $V = E\,r$ para $r \ge a$ es una comprobación rápida: si sus dos
respuestas no la cumplen, hay un error.

**Ruta 2.** $R + T$ tiene que dar exactamente 1. Y no olvide el factor
$n_2/n_1$ en $T$: es el error más frecuente.

**Ruta 3.** Compruebe que $|\Gamma| \le 1$ para cualquier carga pasiva. Y
recuerde que la impedancia de entrada se repite cada media longitud de onda.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Ruta 1: electrostática (Unidad 1) ---
rho_v = 3.0e-6          # densidad de carga volumétrica [C/m^3]
radio_esfera = 0.02     # radio de la esfera a [m]
radio_observacion = 0.05  # distancia de observación r [m]

# --- Ruta 2: interfaz óptica (Unidad 2) ---
n1 = 1.0                # índice del medio de entrada
n2 = 1.5                # índice del medio de salida
campo_incidente = 8.0   # amplitud del campo incidente [V/m]

# --- Ruta 3: línea de transmisión (Unidad 3) ---
Z0 = 50.0                      # impedancia característica [ohm]
ZL = 100.0                     # impedancia de la carga [ohm]
longitud_sobre_lambda = 0.125  # longitud de la línea, en longitudes de onda

## 6. Implementación

### 6.1 Ruta 1 — la esfera cargada

In [ ]:
carga_total = carga_total_esfera_uniforme(radio_esfera, rho_v)
campo = float(campo_esfera_uniforme(radio_observacion, radio_esfera, rho_v))
potencial = float(potencial_esfera_uniforme(radio_observacion, radio_esfera, rho_v))

### 6.2 Ruta 2 — la interfaz

In [ ]:
Gamma_optico = coeficiente_reflexion_normal(n1, n2)
tau_optico = coeficiente_transmision_normal(n1, n2)

campo_reflejado = Gamma_optico * campo_incidente
campo_transmitido = tau_optico * campo_incidente

R = reflectancia_normal(n1, n2)
T = transmitancia_normal(n1, n2)

### 6.3 Ruta 3 — la línea

In [ ]:
Gamma_linea = coeficiente_reflexion(ZL, Z0)
roe = razon_onda_estacionaria(Gamma_linea)
Zin = impedancia_entrada(ZL, Z0, longitud_electrica(longitud_sobre_lambda))

## 7. Resultados numéricos

### Ruta 1 — Unidad 1

In [ ]:
tabla_resultados(
    [
        ("Carga total", "Q", carga_total, "C"),
        ("Campo a la distancia de observación", "|E|", campo, "V/m"),
        ("Potencial a la distancia de observación", "V", potencial, "V"),
        ("Comprobación V / E (debe dar r)", "V/E", potencial / campo, "m"),
        ("Distancia de observación", "r", radio_observacion, "m"),
    ]
)

### Ruta 2 — Unidad 2

In [ ]:
tabla_resultados(
    [
        ("Coeficiente de reflexión", "Gamma", Gamma_optico, "-"),
        ("Coeficiente de transmisión", "tau", tau_optico, "-"),
        ("Campo reflejado", "E_r", campo_reflejado, "V/m"),
        ("Campo transmitido", "E_t", campo_transmitido, "V/m"),
        ("Potencia reflejada", "R", R, "-"),
        ("Potencia transmitida", "T", T, "-"),
        ("Suma", "R + T", R + T, "-"),
    ]
)

### Ruta 3 — Unidad 3

In [ ]:
tabla_resultados(
    [
        ("Coeficiente de reflexión", "Gamma", Gamma_linea.real, "-"),
        ("Módulo de la reflexión", "|Gamma|", abs(Gamma_linea), "-"),
        ("Razón de onda estacionaria", "ROE", roe, "-"),
        ("Impedancia de entrada, real", "Re(Z_in)", Zin.real, "ohm"),
        ("Impedancia de entrada, imaginaria", "Im(Z_in)", Zin.imag, "ohm"),
    ]
)

In [ ]:
print(f"Ruta 1: V/E = {potencial / campo:.6f} m, r = {radio_observacion:.6f} m")
print(f"Ruta 2: R + T = {R + T:.12f}")
print(f"Ruta 3: |Gamma| = {abs(Gamma_linea):.6f}")
print()

# La relación V = E r solo vale fuera de la esfera, donde el campo
# decae como 1/r^2 y el potencial como 1/r. Dentro no tiene por qué
# cumplirse: el ejercicio 1 pide justamente comprobarlo.
if radio_observacion >= radio_esfera:
    assert np.isclose(potencial / campo, radio_observacion), "Falla V = E r"
    print("Ruta 1: V = E r se cumple, como corresponde fuera de la esfera.")
else:
    print("Ruta 1: V = E r NO se cumple, y está bien: el punto de observación")
    print("        quedó dentro de la esfera, donde esa relación no aplica.")

assert np.isclose(R + T, 1.0), "La potencia no se conserva"
assert abs(Gamma_linea) <= 1.0, "Reflexión mayor que 1 en una carga pasiva"
print("Ruta 2: la potencia se conserva.")
print("Ruta 3: la reflexión es menor o igual que 1, como toda carga pasiva.")

## 8. Visualización

Una figura por unidad. Fíjese en que la del centro y la de la derecha
describen el mismo fenómeno —una onda que se topa con un cambio de medio— con
variables distintas.

In [ ]:
fig, (eje_esfera, eje_interfaz, eje_linea) = plt.subplots(1, 3, figsize=(12.0, 3.8))

# Unidad 1: campo de la esfera
r = np.linspace(1.0e-5, 3.0 * radio_esfera, 400)
eje_esfera.plot(r * 100.0, campo_esfera_uniforme(r, radio_esfera, rho_v))
eje_esfera.axvline(radio_esfera * 100.0, color="black", linestyle="--", label="superficie")
eje_esfera.scatter([radio_observacion * 100.0], [campo], color="black", zorder=5)
eje_esfera.set_xlabel("r (cm)")
eje_esfera.set_ylabel("|E| (V/m)")
eje_esfera.set_title("Unidad 1: esfera cargada")
eje_esfera.legend(fontsize=8)

# Unidad 2: reflexión en función del contraste de índices
indices = np.linspace(1.0, 4.0, 300)
eje_interfaz.plot(indices, [reflectancia_normal(n1, n) for n in indices])
eje_interfaz.scatter([n2], [R], color="black", zorder=5, label="caso evaluado")
eje_interfaz.set_xlabel("n_2")
eje_interfaz.set_ylabel("R")
eje_interfaz.set_title("Unidad 2: reflexión en la interfaz")
eje_interfaz.legend(fontsize=8)

# Unidad 3: impedancia de entrada
longitudes = np.linspace(0.0, 0.5, 500)
Zin_curva = impedancia_entrada(ZL, Z0, longitud_electrica(longitudes))
eje_linea.plot(longitudes, abs(Zin_curva))
eje_linea.axhline(Z0, color="black", linestyle=":", label=f"Z_0 = {Z0:.0f} ohm")
eje_linea.scatter([longitud_sobre_lambda], [abs(Zin)], color="black", zorder=5)
eje_linea.set_xlabel("Longitud / lambda")
eje_linea.set_ylabel("|Z_in| (ohm)")
eje_linea.set_title("Unidad 3: línea de transmisión")
eje_linea.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**Ruta 1: la esfera se ve como una carga puntual.** A 5 cm de una esfera de
2 cm de radio, el campo y el potencial cumplen $V = E\,r$ exactamente, que es
la firma del comportamiento $1/r^2$ y $1/r$.

**Ruta 2: se refleja el 4 %.** El mismo resultado de la semana 10. Note que
$\Gamma$ es negativo: la onda reflejada sale invertida.

**Ruta 3: la carga del doble refleja un tercio del campo.** Con $Z_L = 2Z_0$,
$\Gamma = 1/3$ y la ROE es exactamente 2. Es un caso que conviene tener
memorizado: **ROE $= Z_L/Z_0$ cuando la carga es real y mayor que $Z_0$**.

**A un octavo de longitud de onda la impedancia es compleja.** Partiendo de
una carga puramente resistiva, la línea le agrega reactancia. Ésa es
exactamente la propiedad que se explota para adaptar.

**Las tres comprobaciones pasan.** $V = Er$, $R + T = 1$ y
$|\Gamma| \le 1$. Son tres formas distintas de la misma idea: la física tiene
que cerrar.

La primera lleva una condición: solo vale **fuera** de la esfera. Si usted
mueve el punto de observación hacia adentro, el notebook se lo dice en vez de
fallar. Toda comprobación automática tiene un rango de validez, y conviene
saber cuál es.

## 10. Ejercicios para experimentar

            1. En la ruta 1, ponga `radio_observacion = 0.01`, dentro de la esfera. ¿Sigue
               cumpliéndose $V = E\,r$? ¿Por qué no? Mire de nuevo el gráfico.
            2. En la ruta 2, ponga `n2 = 1.0`. ¿Qué pasa con $\Gamma$, $R$ y $T$?
            3. En la ruta 2, ponga `n2 = 3.5` (silicio). ¿Cuánta luz se pierde? Compare
               con el gráfico del centro.
            4. En la ruta 3, ponga `ZL = 25.0`. ¿Cuánto vale la ROE? Compárela con el caso
               `ZL = 100.0` y explique la coincidencia.
            5. En la ruta 3, ponga `longitud_sobre_lambda = 0.25`. ¿Qué impedancia
               obtiene? Verifique $Z_0^2/Z_L$.
            6. Escriba en una hoja las tres fórmulas de $\Gamma$ que aparecen en el curso
               (interfaz óptica, condición de borde, línea de transmisión) y explique en
               una frase por qué son la misma idea.